In [2]:

import os 
os.chdir(os.path.dirname(os.getcwd()))


In [3]:
import ast
import pandas as pd
from sklearn.metrics import f1_score as f1 
from sklearn.metrics import precision_recall_fscore_support as prf

## Obitools

In [7]:


def can_family(tax_ids,database):

    sub = database[database['taxid_ncbi'].isin(tax_ids)]
    
    family = sub['family'].unique()

    if len(family) > 1:
        return 'NC_'
    return family[0]

acc = []
f1_scrores = []
p_score =[]
r_score =[]

OBI_pred_path = 'Obitools/scripts/results/obi3/teleo'

for fold in range(1,7):

    print("-----------------")
    database = pd.read_csv(f'Obitools/scripts/data/teleov2_no_cefe/folds/fold_{fold}/train.csv')

    preds = pd.read_csv(OBI_pred_path +f'/fold_{fold}_test_ecotag3_pred.csv', sep="\t")



    preds['BEST_MATCH_IDS'] = preds['BEST_MATCH_IDS'].apply(ast.literal_eval)  # Convert to list of strings
    preds['BEST_MATCH_TAXIDS'] = preds['BEST_MATCH_TAXIDS'].apply(ast.literal_eval) 

    preds['family'] = preds['BEST_MATCH_TAXIDS'].apply(lambda x: can_family(x,database))

    labels = pd.read_csv(f'Obitools/scripts/data/teleov2_no_cefe/folds/fold_{fold}/test.csv')

    classes = labels['family'].unique()

    preds['ak_family'] = labels['family']


    p,r,f,s = prf(preds["ak_family"], preds["family"], average="macro", labels=classes, zero_division=0)
    print(f'Fold {fold} f1: {f}')
    print(f'Fold {fold} precision: {p}')
    print(f'Fold {fold} recall: {r}')

    
    # preds.to_csv(OBI_pred_path +f'/fold_{fold}_test_ecotag3_pred.csv', sep="\t", index=False)
    





-----------------
Fold 1 f1: 0.47465461798636027
Fold 1 precision: 0.4739410382432297
Fold 1 recall: 0.5295737901907528
-----------------
Fold 2 f1: 0.45334882018659667
Fold 2 precision: 0.4435489973857547
Fold 2 recall: 0.5184015261779448
-----------------
Fold 3 f1: 0.468639516330956
Fold 3 precision: 0.46484546611405203
Fold 3 recall: 0.530454500577492
-----------------
Fold 4 f1: 0.4740957234523496
Fold 4 precision: 0.4715387060832812
Fold 4 recall: 0.5344436006828097
-----------------
Fold 5 f1: 0.4604431932210324
Fold 5 precision: 0.45225582727009034
Fold 5 recall: 0.5242502467788803
-----------------
Fold 6 f1: 0.45210807646477363
Fold 6 precision: 0.44843482768147364
Fold 6 recall: 0.5066849733720649


## DNABERT_2

In [10]:
padnas_macro= []
F = []
for fold in [1,2,3,4,5,6]:
    print("-----------------")
    preds = pd.read_csv(f'DNABert2/experiments/fine_tune_taxa/outputs/teleo_from_mlm_no_cefe/checkpoints/fold_{fold}/predictions.csv')
    
    print("fold",fold)
    p,r,f,s = prf(preds['labels_family'],preds['preds_family'],average='macro', labels=preds['labels_family'].unique(), zero_division=0)
    print('p',p,'r',r,'f',f)

    

-----------------
fold 1
p 0.4642533434341877 r 0.55789263076207 f 0.4836824647697926
-----------------
fold 2
p 0.4481972977970539 r 0.5577167386831203 f 0.4658507572767918
-----------------
fold 3
p 0.4796223957146761 r 0.5675838102955517 f 0.4911660270706371
-----------------
fold 4
p 0.47466540951553826 r 0.5656663083742823 f 0.4893774999603046
-----------------
fold 5
p 0.45564516147997575 r 0.5616006830912543 f 0.47518455491297984
-----------------
fold 6
p 0.48667309766064204 r 0.5706910078846353 f 0.5022582543266843


## MMSeq2


In [15]:

for fold in [1,2,3,4,5,6]:

    preds_df = pd.read_csv(f'/home/auguste/Desktop/eDNA/TeleoClassification/scripts/MMseqs2/result/teleov2_no_cefe/folds/fold_{fold}/tax_out/tax_out_lca.tsv', sep="\t", header=None)
    ground_truth = pd.read_csv(f'/home/auguste/Desktop/eDNA/TeleoClassification/scripts/MMseqs2/teleov2_no_cefe/folds/fold_{fold}/test.csv')
    print("====================")
    print("fold",fold)

    preds_df[0] = preds_df[0].apply(lambda x: x.split('_')[-1]) # remove the prefix

    p,r,f,s = prf(preds_df[3],preds_df[0],average='macro',labels=ground_truth['family'].unique(),zero_division=0)
    print('p',p,'r',r,'f',f)

    


fold 1
p 0.45987804529316245 r 0.4070008982790504 f 0.406857546466756
fold 2
p 0.49543263874680316 r 0.4309593355891265 f 0.43946589586998824
fold 3
p 0.49082541644917993 r 0.44518171677814555 f 0.4451590427375763
fold 4
p 0.5032980010288068 r 0.4519480410961539 f 0.45543532904197825
fold 5
p 0.4746462261252633 r 0.4222421637758673 f 0.4247561992833887
fold 6
p 0.47514933323756847 r 0.4197015121778769 f 0.4233633599076589


In [3]:
import pandas as pd
df = pd.read_csv('Obitools/scripts/data/teleov2_no_cefe/folds/fold_1/train.csv')
label_to_idx = {label: idx for idx, label in enumerate(df['family'].unique())}
idx_to_label = {idx: label for idx, label in enumerate(df['family'].unique())}


#save dict
import json
with open('label_to_idx.json', 'w') as f:
    json.dump(label_to_idx, f)
with open('idx_to_label.json', 'w') as f:
    json.dump(idx_to_label, f)